In [3]:
# @title Step 1: Install Dependencies and Import Libraries
# Install all required packages quietly
!pip install faiss-cpu
import pandas as pd
import numpy as np
import joblib
import faiss
import os
import json
from pathlib import Path

# Scikit-learn imports
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

# Transformer imports
from sentence_transformers import SentenceTransformer
from transformers import T5ForConditionalGeneration, T5Tokenizer

# Colab-specific import for file handling
from google.colab import files

print("✅ All libraries are installed and imported.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 27.7 MB/s eta 0:00:00
✅ All libraries are installed and imported.


In [4]:
# @title Step 2: Upload your Dataset
print("Please upload your 'Dataset_Tweets_Facts_01-Sheet1.csv' file.")
uploaded = files.upload()

# Check if the file was uploaded
if 'Dataset_Tweets_Facts_01-Sheet1.csv' in uploaded:
    DATA_FILE = 'Dataset_Tweets_Facts_01-Sheet1.csv'
    print(f"\n✅ File '{DATA_FILE}' uploaded successfully!")
else:
    print("\n❌ File not uploaded correctly. Please make sure the filename is exactly 'Dataset_Tweets_Facts_01-Sheet1.csv' and try again.")
    DATA_FILE = None

Please upload your 'Dataset_Tweets_Facts_01-Sheet1.csv' file.


Saving Dataset_Tweets_Facts_01-Sheet1.csv to Dataset_Tweets_Facts_01-Sheet1.csv

✅ File 'Dataset_Tweets_Facts_01-Sheet1.csv' uploaded successfully!


In [7]:
# @title Step 3: Define All Pipeline Functions (Corrected & Optimized)
# --- 1. CONFIGURATION ---
ARTIFACTS_DIR = Path("fact_checking_artifacts")
BEST_MODEL_INFO_FILE = ARTIFACTS_DIR / "best_model_info.json"
TFIDF_VECTORIZER_PATH = ARTIFACTS_DIR / "tfidf_vectorizer.joblib"
CLASSIFIER_PATH = ARTIFACTS_DIR / "classifier.joblib"
ST_ENCODER_PATH = ARTIFACTS_DIR / "sentence_transformer_encoder"
EVIDENCE_INDEX_PATH = ARTIFACTS_DIR / "evidence_index.faiss"
EVIDENCE_LIST_PATH = ARTIFACTS_DIR / "evidence_list.pkl"
T5_TOKENIZER_PATH = ARTIFACTS_DIR / "t5_tokenizer"
T5_MODEL_PATH = ARTIFACTS_DIR / "t5_model"

# --- 2. TRAINING & ASSET GENERATION FUNCTIONS ---

def load_and_preprocess_data(file_path):
    print(f"Loading and preprocessing data from {file_path}...")
    df = pd.read_csv(file_path)
    df.dropna(subset=['Tweet', 'Label', 'Evidence'], inplace=True)
    label_counts = df['Label'].value_counts()
    rare_classes = label_counts[label_counts < 2].index
    if not rare_classes.empty:
        df = df[~df['Label'].isin(rare_classes)]
    X, y = df['Tweet'], df['Label']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    print(f"Data split into {len(X_train)} training and {len(X_test)} testing samples.")
    return X_train, X_test, y_train, y_test, df

def train_and_evaluate_models(X_train, y_train, X_test, y_test):
    print("\n--- Training and Evaluating Classifier Models ---")
    models_performance = {}

    print("\n[1/3] Training TF-IDF + Logistic Regression...")
    tfidf_vectorizer = TfidfVectorizer(max_features=5000)
    X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
    X_test_tfidf = tfidf_vectorizer.transform(X_test)
    lr_model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42).fit(X_train_tfidf, y_train)
    accuracy_lr = accuracy_score(y_test, lr_model.predict(X_test_tfidf))
    print(f"  > Accuracy: {accuracy_lr:.4f}")
    models_performance['tfidf_lr'] = {'accuracy': accuracy_lr, 'model': lr_model, 'vectorizer': tfidf_vectorizer, 'type': 'tfidf'}

    print("\n[2/3] Training TF-IDF + SVM...")
    svm_model = SVC(class_weight='balanced', kernel='linear', probability=True, random_state=42).fit(X_train_tfidf, y_train)
    accuracy_svm = accuracy_score(y_test, svm_model.predict(X_test_tfidf))
    print(f"  > Accuracy: {accuracy_svm:.4f}")
    models_performance['tfidf_svm'] = {'accuracy': accuracy_svm, 'model': svm_model, 'vectorizer': tfidf_vectorizer, 'type': 'tfidf'}

    print("\n[3/3] Training Sentence Transformer + Logistic Regression...")
    st_encoder = SentenceTransformer('all-mpnet-base-v2')
    print("  > Creating embeddings (will use GPU if available)...")
    X_train_st = st_encoder.encode(X_train.tolist(), show_progress_bar=True)
    X_test_st = st_encoder.encode(X_test.tolist(), show_progress_bar=True)
    st_lr_model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42).fit(X_train_st, y_train)
    accuracy_st_lr = accuracy_score(y_test, st_lr_model.predict(X_test_st))
    print(f"  > Accuracy: {accuracy_st_lr:.4f}")
    models_performance['st_lr'] = {'accuracy': accuracy_st_lr, 'model': st_lr_model, 'encoder': st_encoder, 'type': 'sentence_transformer'}

    best_model_name = max(models_performance, key=lambda k: models_performance[k]['accuracy'])
    best_model_info = models_performance[best_model_name]
    print(f"\n🏆 Best performing model is '{best_model_name}' with accuracy: {best_model_info['accuracy']:.4f}")

    ARTIFACTS_DIR.mkdir(exist_ok=True)
    joblib.dump(best_model_info['model'], CLASSIFIER_PATH)
    model_type = best_model_info['type']
    if model_type == 'tfidf':
        joblib.dump(best_model_info['vectorizer'], TFIDF_VECTORIZER_PATH)
    else:
        best_model_info['encoder'].save(str(ST_ENCODER_PATH))
    print(f"✅ Saved best classifier and its components to {ARTIFACTS_DIR}")
    with open(BEST_MODEL_INFO_FILE, 'w') as f:
        json.dump({'best_model_type': model_type}, f)

def build_retrieval_and_generative_systems(df):
    print("\n--- Building Supporting Systems ---")
    ARTIFACTS_DIR.mkdir(exist_ok=True)
    print("[1/2] Building evidence retrieval system (FAISS)...")
    retrieval_encoder = SentenceTransformer('all-mpnet-base-v2')
    evidence_list = df['Evidence'].unique().tolist()
    evidence_embeddings = retrieval_encoder.encode(evidence_list, show_progress_bar=True)

    # <<< IMPROVEMENT: Build FAISS index on GPU for faster searching >>>
    embedding_dimension = evidence_embeddings.shape[1]
    cpu_index = faiss.IndexFlatL2(embedding_dimension)
    if torch.cuda.is_available():
        print("  > Building FAISS index on GPU for high-speed search.")
        res = faiss.StandardGpuResources()
        gpu_index = faiss.index_cpu_to_gpu(res, 0, cpu_index)
        gpu_index.add(np.array(evidence_embeddings).astype('float32'))
        faiss.write_index(gpu_index, str(EVIDENCE_INDEX_PATH))
    else:
        print("  > Building FAISS index on CPU.")
        cpu_index.add(np.array(evidence_embeddings).astype('float32'))
        faiss.write_index(cpu_index, str(EVIDENCE_INDEX_PATH))

    pd.Series(evidence_list).to_pickle(EVIDENCE_LIST_PATH)
    print(f"✅ Evidence index with {len(evidence_list)} entries saved.")

    print("\n[2/2] Preparing reason generation model (T5)...")
    model_name = 't5-small'
    tokenizer = T5Tokenizer.from_pretrained(model_name)
    model = T5ForConditionalGeneration.from_pretrained(model_name)
    tokenizer.save_pretrained(T5_TOKENIZER_PATH)
    model.save_pretrained(T5_MODEL_PATH)
    print(f"✅ Generative model '{model_name}' saved.")

def run_training_flow():
    if 'DATA_FILE' not in globals() or not DATA_FILE:
      print("Cannot run training without the data file. Please upload it in Step 2.")
      return
    print("===== STARTING TRAINING AND ASSET GENERATION PIPELINE =====")
    X_train, X_test, y_train, y_test, df = load_and_preprocess_data(DATA_FILE)
    train_and_evaluate_models(X_train, y_train, X_test, y_test)
    build_retrieval_and_generative_systems(df)
    print("\n===== ✅ PIPELINE COMPLETE: All artifacts are saved. =====")

# --- 3. INFERENCE (PREDICTION) CLASS ---
class FactCheckPredictor:
    def __init__(self, artifacts_dir):
        print("--- Loading all pipeline components for prediction ---")
        # <<< IMPROVEMENT: Explicitly set device for model loading >>>
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"  > Using device: {self.device}")

        with open(BEST_MODEL_INFO_FILE, 'r') as f:
            self.model_info = json.load(f)
        self.best_model_type = self.model_info['best_model_type']
        print(f"  > Using classifier type: {self.best_model_type}")

        self.classifier = joblib.load(CLASSIFIER_PATH)
        if self.best_model_type == 'tfidf':
            self.vectorizer = joblib.load(TFIDF_VECTORIZER_PATH)
            # <<< FIX: Load a separate encoder for retrieval when using TF-IDF >>>
            self.retrieval_encoder = SentenceTransformer('all-mpnet-base-v2', device=self.device)
        else: # sentence_transformer
            # <<< FIX: Load the saved encoder and reuse it to save memory/time >>>
            self.classification_encoder = SentenceTransformer(str(ST_ENCODER_PATH), device=self.device)
            self.retrieval_encoder = self.classification_encoder

        self.evidence_index = faiss.read_index(str(EVIDENCE_INDEX_PATH))
        self.evidence_list = pd.read_pickle(EVIDENCE_LIST_PATH).tolist()
        # <<< IMPROVEMENT: Move T5 model to the correct device upon loading >>>
        self.reason_tokenizer = T5Tokenizer.from_pretrained(T5_TOKENIZER_PATH)
        self.reason_model = T5ForConditionalGeneration.from_pretrained(T5_MODEL_PATH).to(self.device)
        print("✅ All components loaded successfully.")

    def predict(self, tweet_text):
        print("\n-------------------------------------------")
        print(f"🔎 Analyzing tweet: \"{tweet_text}\"")

        # 1. Classification: Predict the label
        if self.best_model_type == 'tfidf':
            tweet_vector = self.vectorizer.transform([tweet_text])
            predicted_label = self.classifier.predict(tweet_vector)[0]
            # Encode separately for retrieval
            retrieval_embedding = self.retrieval_encoder.encode([tweet_text], convert_to_numpy=True)
        else: # sentence_transformer
            # <<< IMPROVEMENT: Encode once and reuse for both tasks >>>
            tweet_embedding_np = self.classification_encoder.encode([tweet_text], convert_to_numpy=True)
            predicted_label = self.classifier.predict(tweet_embedding_np)[0]
            retrieval_embedding = tweet_embedding_np

        # 2. Evidence Retrieval
        _, I = self.evidence_index.search(retrieval_embedding, k=1)
        retrieved_evidence = self.evidence_list[I[0][0]]

        # 3. Reason Generation
        prompt = f"""Based on the following information: - Tweet: "{tweet_text}" - Predicted Label: "{predicted_label}" - Relevant Evidence: "{retrieved_evidence}" Write a brief reason explaining why the tweet was given this label. Reason:"""
        input_ids = self.reason_tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True).input_ids.to(self.device)
        outputs = self.reason_model.generate(input_ids, max_length=100, num_beams=5, early_stopping=True)
        generated_reason = self.reason_tokenizer.decode(outputs[0], skip_special_tokens=True)

        print("\n--- Fact-Check Analysis Complete ---")
        print(f"Predicted Label: {predicted_label}")
        print(f"Ranked Evidence: {retrieved_evidence}")
        print(f"Generated Reason: {generated_reason}")
        print("-------------------------------------------")

# Final check to confirm the functions are defined for the notebook
print("✅ All functions defined.")

✅ All functions defined.


In [9]:
# @title ▶️ Run the Full Training Process
run_training_flow()

===== STARTING TRAINING AND ASSET GENERATION PIPELINE =====
Loading and preprocessing data from Dataset_Tweets_Facts_01-Sheet1.csv...
Data split into 1469 training and 368 testing samples.

--- Training and Evaluating Classifier Models ---

[1/3] Training TF-IDF + Logistic Regression...
  > Accuracy: 0.5326

[2/3] Training TF-IDF + SVM...
  > Accuracy: 0.5978

[3/3] Training Sentence Transformer + Logistic Regression...
  > Creating embeddings (will use GPU if available)...


Batches:   0%|          | 0/46 [00:00<?, ?it/s]

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

  > Accuracy: 0.4103

🏆 Best performing model is 'tfidf_svm' with accuracy: 0.5978
✅ Saved best classifier and its components to fact_checking_artifacts

--- Building Supporting Systems ---
[1/2] Building evidence retrieval system (FAISS)...


Batches:   0%|          | 0/57 [00:00<?, ?it/s]

NameError: name 'torch' is not defined

In [ ]:
# @title Load the saved artifacts for prediction
if os.path.exists(ARTIFACTS_DIR):
    predictor = FactCheckPredictor(ARTIFACTS_DIR)
else:
    print("❌ Artifacts not found. Please run the training cell (Step 4) first.")
    predictor = None